In [2]:
# Setup cell — run this first before any other cell
import os
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# Required for async/await in Jupyter
nest_asyncio.apply()

# Load API keys from .env file
load_dotenv()

# Initialize PyRIT memory
from pyrit.memory import SQLiteMemory, CentralMemory
memory = SQLiteMemory()
CentralMemory.set_memory_instance(memory)

# --- Kurumsal pretty print yardımcıları ---
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich import box
    _console = Console()
    _RICH = True
except ImportError:
    _RICH = False


def _render_status(rows, title="Environment Status"):
    """Durum satırlarını kurumsal bir formatta yazdırır."""
    if _RICH:
        table = Table(
            title=title,
            box=box.ROUNDED,
            header_style="bold white on blue",
            title_style="bold cyan",
            show_lines=True,
            expand=False,
        )
        table.add_column("Bileşen", style="bold yellow", no_wrap=True)
        table.add_column("Durum", justify="center")
        table.add_column("Detay", style="dim")

        for name, ok, detail in rows:
            status = "[bold green]✓ READY[/bold green]" if ok else "[bold red]✗ NOT SET[/bold red]"
            table.add_row(name, status, detail)

        _console.print()
        _console.print(Panel(table, border_style="cyan", title="[bold]PyRIT Setup[/bold]", subtitle="Kurumsal Kontrol Paneli"))
        _console.print()
    else:
        # rich yoksa sade fallback
        line = "═" * 62
        print(f"\n╔{line}╗")
        print(f"║ {title:^60} ║")
        print(f"╠{line}╣")
        for name, ok, detail in rows:
            icon = "✓ READY  " if ok else "✗ NOT SET"
            print(f"║ {name:<18} │ {icon:<10} │ {detail:<26} ║")
        print(f"╚{line}╝\n")


# --- Anahtar kontrolü ---
gemini_key = os.getenv("GEMINI_API_KEY", "")
gemini_ready = bool(gemini_key) and gemini_key not in {
    "your-gemini-key-here",
    "AIza_your-gemini-key-here",
}
groq_key = os.getenv("GROQ_API_KEY", "")
groq_ready = bool(groq_key) and groq_key not in {
    "gsk_your-groq-key-here",
    "your-groq-key-here",
}

status_rows = [
    ("Gemini API", gemini_ready, "GEMINI_API_KEY .env içinde tanımlı" if gemini_ready else "GEMINI_API_KEY .env dosyasına ekleyin"),
    ("Groq API",   groq_ready,   "GROQ_API_KEY .env içinde tanımlı"   if groq_ready   else "GROQ_API_KEY .env dosyasına ekleyin"),
    ("Memory",     True,         "SQLiteMemory başlatıldı (CentralMemory)"),
]

_render_status(status_rows)

╭────────────────────────────────────────────────── PyRIT Setup ──────────────────────────────────────────────────╮
│                         Environment Status                                                                      │
│ ╭────────────┬─────────┬─────────────────────────────────────────╮                                              │
│ │ Bileşen    │  Durum  │ Detay                                   │                                              │
│ ├────────────┼─────────┼─────────────────────────────────────────┤                                              │
│ │ Gemini API │ ✓ READY │ GEMINI_API_KEY .env içinde tanımlı      │                                              │
│ ├────────────┼─────────┼─────────────────────────────────────────┤                                              │
│ │ Groq API   │ ✓ READY │ GROQ_API_KEY .env içinde tanımlı        │                                              │
│ ├────────────┼─────────┼─────────────────────────────────────────┤                                              │
│ │ Memory     │ ✓ READY │ SQLiteMemory başlatıldı (CentralMemory) │                                              │
│ ╰────────────┴─────────┴─────────────────────────────────────────╯                                              │
╰──────────────────────────────────────────── Kurumsal Kontrol Paneli ────────────────────────────────────────────╯

In [4]:
import os
import nest_asyncio
from dotenv import load_dotenv

from pyrit.prompt_target import OpenAIChatTarget
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.models import Message, MessagePiece
from pyrit.memory import SQLiteMemory, CentralMemory

nest_asyncio.apply()
load_dotenv()

# PyRIT memory kurulumu
try:
    CentralMemory.get_memory_instance()
except ValueError:
    CentralMemory.set_memory_instance(SQLiteMemory())

# Rich çıktıları
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich.rule import Rule
    from rich import box

    _console = Console()
    _RICH = True
except ImportError:
    _RICH = False


GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_ENDPOINT = (
    "https://generativelanguage.googleapis.com/v1beta/openai/"
)

gemini_api_key = os.environ["GEMINI_API_KEY"]

gemini_target = OpenAIChatTarget(
    model_name=GEMINI_MODEL,
    endpoint=GEMINI_ENDPOINT,
    api_key=gemini_api_key,
    max_completion_tokens=300,  # max_tokens yerine
    temperature=0.7,
    top_p=0.95,
)

_config = [
    ("model_name", GEMINI_MODEL),
    ("endpoint", GEMINI_ENDPOINT),
    ("max_completion_tokens", "300"),
    ("temperature", "0.7"),
    ("top_p", "0.95"),
    ("api_key", "GEMINI_API_KEY (env)"),
]

if _RICH:
    cfg_table = Table(
        box=box.SIMPLE_HEAD,
        show_header=False,
        title="[bold cyan]Gemini Chat Target - Config[/bold cyan]",
    )
    cfg_table.add_column("Ayar", style="bold yellow", no_wrap=True)
    cfg_table.add_column("Değer", style="white")

    for key, value in _config:
        cfg_table.add_row(key, str(value))

    _console.print()
    _console.print(Panel(cfg_table, border_style="cyan"))
else:
    print("\n=== Gemini Chat Target - Config ===")
    for key, value in _config:
        print(f"  {key:<24}: {value}")
    print()


prompt_list = [
    "What is SQL injection in one sentence?",
    "What is prompt injection in one sentence?",
    "What is the role of Data Science?",
]

normalizer = PromptNormalizer()
responses = []

for prompt_text in prompt_list:
    message = Message(
        message_pieces=[
            MessagePiece(
                role="user",
                original_value=prompt_text,
            )
        ]
    )

    response = await normalizer.send_prompt_async(
        message=message,
        target=gemini_target,
    )
    responses.append(response)


def _extract_assistant(response):
    for piece in response.message_pieces:
        if piece.role == "assistant":
            return str(piece.original_value)

    return "<yanıt yok>"


if _RICH:
    _console.print()
    _console.print(
        Rule(
            f"[bold green]Gönderilen {len(prompt_list)} prompt - "
            f"Alınan {len(responses)} yanıt[/bold green]"
        )
    )
    _console.print()

    for index, response in enumerate(responses, 1):
        question = prompt_list[index - 1]
        answer = _extract_assistant(response)

        body = (
            f"[bold yellow]S:[/bold yellow] {question}\n\n"
            f"[bold green]C:[/bold green] {answer}"
        )

        _console.print(
            Panel(
                body,
                title=f"[bold]Sorgu #{index}[/bold]",
                border_style="blue",
                expand=False,
            )
        )
        _console.print()
else:
    print(
        f"\n--- {len(prompt_list)} prompt gönderildi, "
        f"{len(responses)} yanıt alındı ---\n"
    )

    for index, response in enumerate(responses, 1):
        print(f"[{index}] S: {prompt_list[index - 1]}")
        print(f"    C: {_extract_assistant(response)}")
        print()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                             Gemini Chat Target - Config                                                         │
│                                                                                                                 │
│   model_name              gemini-3.5-flash-lite                                                                 │
│   endpoint                https://generativelanguage.googleapis.com/v1beta/openai/                              │
│   max_completion_tokens   300                                                                                   │
│   temperature             0.7                                                                                   │
│   top_p                   0.95                                                                                  │
│   api_key                 GEMINI_API_KEY (env)                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The response was truncated because it reached the token limit (finish_reason='length'). Reasoning models consume tokens on hidden reasoning in addition to the visible answer, so a low max_completion_tokens can truncate or empty the response. Increase max_completion_tokens if you expected complete content.


────────────────────────────────────── Gönderilen 3 prompt - Alınan 3 yanıt ───────────────────────────────────────

╭─────────────────────────────────────────────────── Sorgu #1 ────────────────────────────────────────────────────╮
│ S: What is SQL injection in one sentence?                                                                       │
│                                                                                                                 │
│ C: SQL injection is a cyberattack where malicious SQL code is inserted into input fields to manipulate a        │
│ database, allowing attackers to view, modify, or delete unauthorized data.                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Sorgu #2 ────────────────────────────────────────────────────╮
│ S: What is prompt injection in one sentence?                                                                    │
│                                                                                                                 │
│ C: Prompt injection is a vulnerability where an attacker manipulates an AI's input so that it ignores its       │
│ original instructions and executes unauthorized commands instead.                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Sorgu #3 ────────────────────────────────────────────────────╮
│ S: What is the role of Data Science?                                                                            │
│                                                                                                                 │
│ C: At its core, the role of **Data Science** is to **extract actionable meaning, insights, and value from       │
│ data.**                                                                                                         │
│                                                                                                                 │
│ In today’s world, organizations generate massive amounts of data (from user clicks, financial transactions, IoT │
│ sensors, social media, etc.). However, raw data is unstructured and useless on its own. Data science bridges    │
│ the gap between raw data and informed decision-making.                                                          │
│                                                                                                                 │
│ Here is a breakdown of the primary roles and functions of Data Science across different domains:                │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. Enabling Better Decision-Making                                                                          │
│ Traditionally, business decisions were made using intuition and historical reporting (looking backward). Data   │
│ science allows organizations to look **forward**. By using predictive modeling and statistical analysis,        │
│ leaders can make data-driven decisions regarding marketing strategies, financial investments, and operational   │
│ bottlenecks.                                                                                                    │
│                                                                                                                 │
│ ### 2. Automating Processes and Creating Smart Products                                                         │
│ Data science is the engine behind Artificial Intelligence (AI) and Machine Learning (ML). It is used to build   │
│ systems that can learn from data and perform tasks automatically. Examples include:                             │
│ *   **Recommendation Engines:** Netflix, Amazon, and Spotify suggesting what you should watch, buy, or listen   │
│ to next.                                                                                                        │
│ *   **Fraud Detection:** Banks instantly spotting unusual credit card transactions to prevent theft.            │
│ *   **Autonomous Vehicles:** Self-driving cars processing real-time sensor data to navigate safely.             │
│                                                                                                                 │
│ ### 3. Solving Complex Problems                                                                                 │
│ Data scientists tackle open-ended problems that require deep investigation                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯